# convert

Turns Hive someone else wrote, one query or a batch of scripts, into your own code: each table the scripts write becomes a Saved table in your Dev Database, with its Table folder, and what only you can decide of it, a Decision, answered in the decisions cell.

## Terms

- **Step**: each numbered heading below, with the code cell under it. Its first line says its kind: it changes only files, or only prints; it is a call to the warehouse, yours to run; or it is a decision, yours to make.
- **Names cell**: the code cell after the one that finds the Project. It holds this Workflow's inputs, the start materials you fill in once, each with what to write in its comment; each step's call reads them.
- **Decisions cell**: `DECISIONS`, the code cell after the names cell, where you answer each open Decision a step waits on, by the table's name.
- **Hive**: the SQL your warehouse runs.
- **Batch**: the scripts converted together, under one name.
- **Saved table**: a table in the warehouse that only your own code writes, its files in a folder of its own.
- **Dev Database**: the database where you create and write tables of your own, each built, written and compared there first: `DEV` in admin/databases.py.
- **Table folder**: the folder in tables/ that holds everything about one Saved table: its Table notebook, its past lineage, and its Hive.
- **Decision**: one thing about one table that only you decide, such as its location or its Key; the table's files say whether it is answered yet.
- **Table reference**: a script in references/ describing one table: its columns and their types, its Partitions and its Key.
- **Reads itself**: a table whose own script reads it, such as a running total, seeded once from its original before its first write.
- **Original**: the table a converted script writes, in the warehouse already, which its Saved table copies; for a query alone, such as one Copilot wrote from a workbook, the rows the query gives.
- **Session setting**: a `SET` line sent before a Statement, which it needs to run.
- **Partition**: the rows of a table that share one value of each column it is partitioned by.
- **Table notebook**: the notebook in a Table folder whose cells create, preview and write its Saved table; its logic cell is the one whose first line is `# logic`.
- **Key**: the columns whose values name exactly one row of a table.
- **Statement**: a query your code builds, which becomes one Hive string.

## Before step 1

Fill in the names cell below and save the notebook: each call reads it, and so do `project.check()`, which names the next step, and `project.do_next()`, the cell before the last.

## Decision rules

- If you run the steps yourself, run each step's cell in turn, in order.
- If you want them run for you, run the last cell, `project.run_workflow(...)`, in place of their cells. It runs every step left in turn, up to the first only you can do, and says where it stopped: go on from that step.
- If you want only the steps that change files run for you, run `project.do_next()`, the cell before the last. It runs them up to the first it can't.

## Self-check

Once the last step is done, each of these holds:

- Each table the scripts write has its Table folder in tables/ and its Table reference in references/.
- Each table's `conversion/checklist.md` says None under Open Decisions.
- admin/sent.csv records each table created and written, and each that reads itself seeded.
- Each table's `conversion/compare.md` says it passed, and the batch's folder is in admin/workflows/convert/finished/.
- `project.check()` names no next step of this batch.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[2]))  # the Project: three up

from sqlglot_composer import admin, hive_settings, project, run_execute, run_query, show_hive
from warehouse import query, execute

In [ ]:
NAME = "weekly_fails"  # the batch's name, plain lowercase, such as "weekly" (or, for the last cell alone, a list of batches converted already)
FILES = ["weekly_fails.hql"]  # the scripts' files, from this folder, such as ["daily.hql"], or [] to read HIVE
HIVE = None  # the Hive as text, such as """SELECT ...""", or None to read FILES
DATABASE = 'default'  # the database of a table the Hive names without one
RENAME = {"mart.team_fails": "team_fails"}  # a name of your own for a table's copy in dev, by the table's full name: mart_dev.team_fails
STAGING = {}  # each staging table convert names, kept or folded, such as {"ops_reports.failed": "keep"}: left out, a temporary or DROPped one is folded, any other kept


In [ ]:
DECISIONS = {"mart_dev.team_fails": {"location": "/data/team/mart_dev.db/team_fails"}}  # your answer to each open Decision, by the table's name as
# project.check() writes it, such as
# {"mart_dev.weekly_ops": {"location": "/data/team/mart_dev.db/weekly",
#                          "key": ["job_id"], "types": {"fails": "bigint"}}}
# "key": None says no columns name one row. project.check() names each one open;
# a Key proposed from a query is made, but an answer here changes it


## 1. Convert the scripts: each table they write a Saved table in dev

It is a call to the warehouse, yours to run.

- **Goal:** a Table folder and a Table reference for each table the scripts write, each a Saved table in your Dev Database.
- **Action:** Run the cell below. It reads DESCRIBE through your query, and writes only files.
- **Done when** admin/workflows/convert/ holds the batch's folder, under its name, with its batch.md, which convert writes last; then step 2.

In [ ]:
project.convert(NAME, HIVE, files=FILES, query=query, database=DATABASE,
                rename=RENAME, staging=STAGING)

## 2. Answer each table's open Decisions in the decisions cell

It is a decision, yours to make.

- **Goal:** an answer to each open Decision of the batch's tables.
- **Action:** Write each answer in the decisions cell, `DECISIONS`, by the table's name as `project.check()` writes it, then save the notebook. Each table's `conversion/checklist.md`, in its Table folder, lists its open Decisions.
- **Done when** the decisions cell answers each Decision `project.check()` names for this batch; then step 3.

## 3. Write each answer into its table's Table reference

It changes only files, so project.do_next() runs it for you.

- **Goal:** each answer written into its table's Table reference.
- **Action:** Run the cell below.
- **Done when** each table's `conversion/checklist.md` says None under Open Decisions; then step 4.

In [ ]:
project.write_decisions(DECISIONS)

## 4. Create each table in dev

It is a call to the warehouse, yours to run.

- **Goal:** each table of the batch made in your Dev Database.
- **Action:** Run the cell below. It sends each table's `create()` through your execute.
- **Done when** admin/sent.csv records a create of each table; then step 5.

In [ ]:
for table in project.batch_tables(NAME):
    run_execute(project.load(table).create(), execute)

## 5. Seed each table that reads itself, once, from its original

It is a call to the warehouse, yours to run.

- **Goal:** each table that reads itself holding its original's rows, once.
- **Action:** Run the cell below. It sends the Session setting their Partitions need, then each such table's `seed()`, through your execute. If no table of the batch reads itself, this step is done once step 4 is.
- **Done when** admin/sent.csv records the seed of each table that reads itself; then step 6.

In [ ]:
run_execute(hive_settings("dynamic_partitions"), execute)
for table in project.batch_tables(NAME, seeded=True):
    run_execute(project.load(table).seed(), execute)

## 6. Write each table in dev

It is a call to the warehouse, yours to run.

- **Goal:** each table's rows written in your Dev Database.
- **Action:** Run the cell below. `project.run_tables` sends each table's `write()` through your execute.
- **Done when** admin/sent.csv records each table written; then step 7.

In [ ]:
project.run_tables(tables=project.batch_tables(NAME), execute=execute)

## 7. Compare each table with its original

It is a call to the warehouse, yours to run.

- **Goal:** each table's rows checked against its original's.
- **Action:** Run the cell below. `project.compare_batch` reads both through your query and writes each table's result in its `conversion/compare.md`.
- **Done when** each table's `conversion/compare.md` says it passed, and the batch's folder has moved to admin/workflows/convert/finished/; the Workflow is done: go through the Self-check at the top.

In [ ]:
project.compare_batch(NAME, query=query)

In [ ]:
project.do_next()

In [ ]:
# Every step left, in place of their cells, up to the first only you can do.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("convert", query=query, execute=execute)